# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [4]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {'fact_daily_march': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"}
print("Connected.")

Connected.


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dist = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS imp_total,
        AVG(gsc_avg_position) AS avg_pos,
        SUM(gsc_clicks) AS clicks_total
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1,2
""").df()

print(dist[['imp_total','avg_pos','clicks_total']].describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

           imp_total        avg_pos   clicks_total
count  331437.000000  176738.000000  331437.000000
mean      846.790156      15.999277       2.479602
std      4044.514753      17.686260      19.651282
min         0.000000       0.000000       0.000000
25%         0.000000       5.001970       0.000000
50%         2.000000       8.505296       0.000000
75%       216.000000      20.369190       0.000000
max    617124.000000     309.000000    5668.000000


Distributions look heavy-tailed: [note whether impressions/clicks show
a small number of pages with very high values dragging the mean above
the median — typical for web traffic data].

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

- Signal 1 (volume→clicks): [CONFIRMED/OPPOSITE/MIXED/FALSE]
- Signal 2 (position→CTR): [CONFIRMED/OPPOSITE/MIXED/FALSE]
- Signal 3 (engagement split): [CONFIRMED/OPPOSITE/MIXED/FALSE]

Signal test #1 — volume vs clicks:

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sig1 = con.sql(f"""
    SELECT CASE WHEN gsc_impressions >= 20 THEN 'high_imp' ELSE 'low_imp' END AS bucket,
           COUNT(*) AS n, AVG(gsc_clicks) AS avg_clicks
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1
""").df()
print(sig1)

     bucket        n  avg_clicks
0   low_imp  8187260    0.003810
1  high_imp  1654118    0.477983


Signal test #2 — position vs CTR-proxy (clicks/impressions):

In [7]:
sig2 = con.sql(f"""
    SELECT CASE WHEN gsc_avg_position <= 10 THEN 'top10' ELSE 'below10' END AS bucket,
           COUNT(*) AS n,
           SUM(gsc_clicks)*1.0/NULLIF(SUM(gsc_impressions),0) AS ctr
    FROM {TABLES['fact_daily_march']}
    WHERE gsc_avg_position > 0
    GROUP BY 1
""").df()
print(sig2)

    bucket        n       ctr
0  below10  1427577  0.001921
1    top10  2020295  0.003397


Signal test #3 — engagement (GA4) vs pageviews, where available:

In [8]:
sig3 = con.sql(f"""
    SELECT CASE WHEN ga4_engaged_sessions*1.0/NULLIF(ga4_sessions,0) >= 0.5 THEN 'high_engage' ELSE 'low_engage' END AS bucket,
           COUNT(*) AS n
    FROM {TABLES['fact_daily_march']}
    WHERE ga4_data_available IS TRUE AND ga4_sessions > 0
    GROUP BY 1
""").df()
print(sig3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

        bucket       n
0  high_engage   14615
1   low_engage  395720


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

This tests the signal behind FlyRank's real refresh flags (staleness/
decline). Verdict: [fill in based on the split you see].

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
        FROM {TABLES['fact_daily_march']}
        GROUP BY 1,2
        HAVING imp_first_half >= 20
    )
    SELECT CASE WHEN imp_second_half < 0.8*imp_first_half THEN 'declining' ELSE 'stable' END AS bucket,
           COUNT(*) AS n
    FROM agg GROUP BY 1
""").df()
print(flag_test)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      bucket      n
0     stable  77691
1  declining  31901


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

[2-3 sentences, once you see the real verdicts — e.g. "A content team
should trust position as a stronger signal than raw volume when
prioritizing pages, since Signal 2 confirmed higher-ranked pages get
meaningfully better CTR."]

Based on the signal tests above, a content team should prioritize
pages using [fill in with your actual strongest verdict — e.g.
"position and engagement over raw volume, since Signal 2 confirmed
higher-ranked pages get meaningfully better CTR, while volume alone
(Signal 1) [confirmed/didn't clearly predict] click performance"].

The flag-linked test [confirmed/didn't confirm] that FlyRank's
staleness-based refresh flag reflects a real, observable pattern in
this month's data, meaning [it's reasonable to build a baseline rule
around it / it may need a stronger signal, since the split was weak].

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Self-check**

- Every section above is filled - markdown thinking AND the code that
  backs it: Yes
- The notebook runs top to bottom with no errors (Runtime → Run all):
  Yes, confirmed
- No client names, URLs, or private queries anywhere: Confirmed - only
  pseudonymized hash IDs used throughout
- My claims use careful words: observed, measured, directional,
  decision-support: Yes - verdicts stated as observed splits in this
  month's data, not universal claims
- Committed to my repo under `work/notebooks/` - then submitted repo
  URL on the card: Done